## Problem Statement

### Title: R&D Literature Research Agent — Automated Multi-Source Research Brief Generation Using LangChain Tool-Calling Agents

### Problem:
##### Enterprise R&D, innovation, and technical due-diligence teams routinely need to research emerging topics before making product or investment decisions. This process typically requires manually searching multiple disconnected sources — general knowledge bases like Wikipedia for background context, and specialized repositories like PubMed or Arxiv for recent peer-reviewed findings — then manually synthesizing scattered information into a structured, decision-ready summary. This is time-consuming, inconsistent across researchers, and difficult to scale across the many topics a team needs to track.

##### Objective:
 ##### Design and implement an AI agent that automates this research workflow end-to-end. The agent should:

##### Accept a natural-language research query from a user
##### Autonomously decide which external tools/APIs to call (background knowledge vs. recent research literature) and in what order
##### Integrate with real external systems via API calls (Wikipedia, PubMed) using LangChain's tool-calling framework
##### Adapt its search strategy if initial results are insufficient or off-topic (agentic re-querying)
##### Return a validated, structured output (not free-form text) — a machine-readable JSON brief containing background, key findings, recent research citations, and open questions — that can be directly consumed by downstream systems (dashboards, reports, databases)

In [1]:
# Install required packages for the LangChain research agent

! pip install -U langchain langchain-openai langchain-community wikipedia python-dotenv xmltodict


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Core Python
import os

# Environment variables
from dotenv import load_dotenv

# LangChain agent
from langchain.agents import create_agent

# LangChain OpenAI-compatible chat model
from langchain_openai import ChatOpenAI

# Wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# pubmed document loader
from langchain_community.tools import PubmedQueryRun
from langchain_community.utilities import PubMedAPIWrapper

# LangChain tool decorator
from langchain.tools import tool

# Structured output
from pydantic import BaseModel, Field

C:\Users\smart\AppData\Local\Temp\ipykernel_18636\2255575937.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [3]:
# Load variables from .env

load_dotenv()

# Get OpenRouter API key
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# Check whether the key is available
print("OpenRouter API Key loaded:", bool(openrouter_api_key))

OpenRouter API Key loaded: True


In [4]:
# Verify OpenRouter API key
if not openrouter_api_key:
    raise ValueError('OPENROUTER_API_KEY not found in .env file')
print('OpenRouter API Key verified successfully.')


OpenRouter API Key verified successfully.


In [5]:
llm = ChatOpenAI(
    model="stealth/ox-alpha",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
    temperature=0
)

print("LLM configured successfully")

LLM configured successfully


In [6]:
# Set a proper User-Agent so Wikipedia's servers don't reject our requests with 403 Forbidden
# (Wikipedia now requires this to identify who is calling their API)
#import wikipedia
#wikipedia.set_user_agent("KIET-Agentic-AI-Class/1.0 ([jogipraveen9122@gmail.com])")

# Wikipedia configuration

wiki_wrapper = WikipediaAPIWrapper(
    top_k_results=2,
    doc_content_chars_max=800
)

wiki_tool = WikipediaQueryRun(
    api_wrapper=wiki_wrapper
)

print("Wikipedia tool ready")

Wikipedia tool ready


In [7]:
wiki_result = wiki_tool.invoke(
    "Retrieval-augmented generation"
)

print(wiki_result)

Page: Retrieval-augmented generation
Summary: Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems.
RAG improves LLMs by incorporating information retrieval before 


In [8]:
pubmed_wrapper = PubMedAPIWrapper(
    top_k_results=3
)

pubmed_tool = PubmedQueryRun(
    api_wrapper=pubmed_wrapper
)

print("PubMed tool created successfully")

PubMed tool created successfully


In [9]:
pub = pubmed_tool.invoke(
    "paracetmol"    
)

In [10]:
pub

'PMID: 8732071\nPublished: --\nTitle: [Determination of paracetmol caffeine and ephedrine hydrochloride in likeshu capsules].\nCopyright Information: \nSummary::\nParacetamol (P), caffeine (C) and ephedrine hydrochloride (EH) are there physiological active component in Likeshu capsules. We have proposed the determination method of these three components using extractive separation-spectrophotometry. Only one amount weighed of sample is required to determine them. The recoveries of P, C and EH were 97.51 +/- 0.64% (n = 9), 98.24 +/- 1.04% (n = 8) and 98.58 +/- 1.09% (n = 9) respectively. The results of determination of samples agreed with those of the HPLC method. This method is sensitive, accurate and simple. The other ingredients of Likeshu capsules do not interfere.'

In [11]:
tools = [ pubmed_tool]

In [12]:
from pydantic import BaseModel, Field


class ResearchBrief(BaseModel):
    """Structured format for a research brief."""

    topic: str = Field(
        description="The main topic being researched."
    )

    background: str = Field(
        description="A concise explanation and background of the topic."
    )

    recent_research: list[str] = Field(
        description="Important research findings, papers, or studies related to the topic."
    )

    key_findings: list[str] = Field(
        description="The most important findings from the research."
    )

    open_questions: list[str] = Field(
        description="Important unresolved questions, limitations, or areas for future research."
    )

In [13]:
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=llm,
    tools=[pubmed_tool],

    system_prompt=(
        "You are an R&D research assistant. "
        "Use the wikipedia tool for background and definitions. "
        "Use the pubmed tool for recent research papers and clinical/scientific findings. "
        "Always check both tools before writing your final answer. "
        "Clearly separate background information from recent research findings, "
        "and note any open or unresolved questions in the field."
    ),

    response_format=ToolStrategy(ResearchBrief)
)

print("Agent created successfully")

Agent created successfully


In [14]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what is paracetamol?"
        }
    ]
})

In [15]:
result  # print the structured output in a readable format

{'messages': [HumanMessage(content='what is paracetamol?', additional_kwargs={}, response_metadata={}, id='285451d1-e5cd-4396-ad89-42522b965958'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': None, 'model_provider': 'openai', 'model_name': 'stealth/ox-alpha', 'system_fingerprint': None, 'id': 'gen-1787547616-hvT6BzrvgmUcfBPun32x', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a03224-01e0-7ab2-a093-e7e38b9b3147-0', tool_calls=[], invalid_tool_calls=[])],
 'structured_response': None}